<a href="https://colab.research.google.com/github/SyChen94/colab/blob/main/defuse.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!git clone https://github.com/chunlinli/defuse.git

Cloning into 'defuse'...
remote: Enumerating objects: 447, done.
remote: Counting objects: 100% (40/40), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 447 (delta 22), reused 20 (delta 9), pack-reused 407 (from 1)
Receiving objects: 100% (447/447), 68.53 MiB | 19.04 MiB/s, done.
Resolving deltas: 100% (111/111), done.


In [4]:
!cd defuse

In [11]:
%cd /content/defuse
!ls
!find . -maxdepth 3 -name "setup.py" -o -name "pyproject.toml"


/content/defuse
defuse		     LICENSE		       simulation_hub_small.csv
defuse.png	     README.md		       simulation_random_large.csv
environment.yml      setup.py		       simulation_random_small.csv
example_large.ipynb  simulation
example_small.ipynb  simulation_hub_large.csv
./setup.py


In [12]:
%cd /content/defuse

!python -m pip install -U pip setuptools wheel
!python -m pip install -e .


/content/defuse
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 42.8 MB/s eta 0:00:00
  Attempting uninstall: wheel
    Found existing installation: wheel 0.45.1
    Uninstalling wheel-0.45.1:
      Successfully uninstalled wheel-0.45.1
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
Obtaining file:///content/defuse
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... 

In [14]:
! pip install igraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 31.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [igraph]


In [19]:
%cd /content/defuse/defuse
!sed -n '1,120p' defuse.py

/content/defuse/defuse
import torch
import numpy as np
from sklearn.linear_model import LassoLarsIC
from scipy.stats import anderson
from defuse.feature import feature_list, feature_vector_score
from defuse.trainer import DefuseTrainer
from defuse.defusenet import DefuseNet

class Defuse:
    def __init__(self, model_config=None, train_config=None):

        self.model_config = model_config if model_config else {
            'hidden_struct': [50],
            'model_type': 'mlp',
            'activation': 'sigmoid',
            'alpha': 0.01,
            'fit_measure': 'anderson',
            'threshold': 0.05
        }
        self.train_config = train_config if train_config else {
            'n_epochs': [500, 3500],
            'penalty_type': ['l1', 'tlp'],
            'penalty_param': [1e-4, 5e-2]
        }

    def _test_goodness_of_fit(self, Z, de_ind):
        '''
        Test goodness of fit for the descendants.

        Args:
        Z: [n, d] numpy array. Residual data.
    

In [17]:
!sed -i "s/LassoLarsIC('bic', normalize=False)/LassoLarsIC('bic')/g" defuse.py

In [21]:
%cd /content/defuse/defuse
!grep -n "normalize" defuse.py | head

/content/defuse/defuse


In [23]:
import numpy as np
from numpy.linalg import solve, eigvals

from defuse.defuse import Defuse
from defuse.utils import set_random_seed, count_accuracy

# -----------------------------
# 1) Your-R-style DAG generator
# -----------------------------
def spectral_radius(B):
    return float(np.max(np.abs(eigvals(B))))

def make_stable_beta(B, radius_max=0.8):
    r = spectral_radius(B)
    if np.isfinite(r) and r > 0 and r >= radius_max:
        B = (radius_max / r) * B
    return B

def simulate_dag_r_style(p, density=0.2, a=0.3, b=0.8, radius_max=0.8, rng=None):
    rng = np.random.default_rng() if rng is None else rng
    pi_order = rng.permutation(p)  # 0..p-1
    B = np.zeros((p, p), dtype=float)

    for ii in range(p - 1):
        for jj in range(ii + 1, p):
            if rng.random() < density:
                u = pi_order[ii]
                v = pi_order[jj]
                B[u, v] = rng.uniform(a, b) * rng.choice([-1.0, 1.0])

    B = make_stable_beta(B, radius_max=radius_max)

    # adjacency A for evaluation (directed)
    A = (np.abs(B) > 1e-12).astype(int)
    np.fill_diagonal(A, 0)
    return A, B, pi_order

# -----------------------------
# 2) Your-R-style SEM with hidden confounders
#    Y = (H Phi + E) (I - B)^{-1}
# -----------------------------
def simulate_Y_from_B_confounded(B, n=500, s=5, phi_strength=0.7, noise_var=0.5, rng=None):
    rng = np.random.default_rng() if rng is None else rng
    p = B.shape[0]
    H = rng.standard_normal(size=(n, s))
    Phi = rng.normal(loc=0.0, scale=phi_strength, size=(s, p))
    E = rng.normal(loc=0.0, scale=np.sqrt(noise_var), size=(n, p))

    IB_inv = solve(np.eye(p) - B, np.eye(p))  # (I-B)^{-1}
    Y = (H @ Phi + E) @ IB_inv
    return Y

# -----------------------------
# 3) Run once
# -----------------------------
set_random_seed(1110)
rng = np.random.default_rng(1110)

p = 20
n = 500
density = 0.2
s = 5
phi2 = 0.5
phi_strength = np.sqrt(phi2)  # IMPORTANT: your paper uses phi^2; code uses sd
noise_var = 0.5

A_true, B_true, pi_true = simulate_dag_r_style(
    p=p, density=density, a=0.3, b=0.8, radius_max=0.8, rng=rng
)

Y = simulate_Y_from_B_confounded(
    B_true, n=n, s=s, phi_strength=phi_strength, noise_var=noise_var, rng=rng
)

m = Defuse()
A_est, _ = m.fit(Y, verbose=False)

result = count_accuracy(A_true, A_est)
print(result)
print("spectral radius(B_true) =", spectral_radius(B_true))


TypeError: LassoLarsIC.__init__() got an unexpected keyword argument 'normalize'